### 2.4 Vector Projection - The Geometric Interpretation

#### 2.4.1 Projection onto a Reference Direction

- A **vector** can be thought of as an **arrow** described by two of its properties
  - **magnitude** - how long it is 
  - **direction** - where it points

- To decompose a vector into its basic components, we **project it along the directions** of all the dimensions it exists in - think of it like the vector casting its **"shadow"** onto the x and y reference axes - this is how we break the vector down into its x and y components

- The **length** of each "shadow" *is* its **component** for that dimension and reveals how much each dimension **contributes** to the original vector as a whole - we designate these "shadows" as the vector's **basic components** since they 
  - **Can be combined in any order** to reconstruct the original - when rebuilding them tip-to-tail, if you start with x first and then y, or y first then x, regardlessly, you still end up with the original  
  - **Cannot be described in terms of each other** - geometrically x and y are at right angles, meaning any change in value for the x component does not affect, and goes completely unnoticed by the y component

In [ ]:
# §2.4 Figure 1 — components_recombine_either_order (per D-08, 1x3 mixed Figure)
#
# col 0 — StaticPanel:  xy projection of a = (2, 3) into its x and y components.
# col 1 — DynamicPanel: reconstruction order — origin -> +x -> a, then origin -> +y -> a (6 frames).
# col 2 — DynamicPanel: orthogonality — a stays anchored at (2, 3); a' sweeps x in
#                       [+2, +1, 0, -1, -2] while a'.y stays at 3 (5 frames).
#
# Run under `%matplotlib widget` (ipympl) to see the live animation loop. The
# `%matplotlib inline` fallback renders frame 0 only.

# --- import plumbing ----------------------------------------------------------
import os
import sys

_here = os.getcwd()
# Notebook is at src/subshader/dsp/dsp.ipynb; dsplot lives at <repo>/research/dsplot/.
# Resolve research/ relative to whichever cwd the kernel happens to use.
for candidate in (
    os.path.join(_here, "research"),                                      # cwd == repo root
    os.path.abspath(os.path.join(_here, "..", "..", "..", "research")),   # cwd == src/subshader/dsp
):
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        break

# %matplotlib widget   # uncomment when running interactively for animation loop
# %matplotlib inline   # fallback — renders frame 0 only

import numpy as np
import matplotlib.pyplot as plt

import dsplot
from dsplot import (
    Annotation,
    Dropline,
    DynamicPanel,
    Figure,
    StaticPanel,
    Vector,
    VectorComponents,
    style,
)
from dsplot.figures.foundation_constants import A, A_PRIME, FOUND_LIM

# Paint the Jupyter cell-output container dark to match the figure bg.
dsplot.apply_jupyter_dark()


# --- col 0: StaticPanel — xy projection of a ----------------------------------
# Bold a in PRIMARY_COLOR, neutral component arrows along x and y, droplines
# from the tip to each axis, and labels on each component.
def _static_xy_projection_panel() -> StaticPanel:
    panel = StaticPanel(
        title="Projection onto x and y",
        subtitle=r"$\vec{a}$ decomposed into $a_x$, $a_y$",
        lim=FOUND_LIM,
        axis_style="arrow",
        axis_labels=True,
        show_border=False,
    )
    ax_val, ay_val = A
    panel.add(Dropline(start=(ax_val, ay_val), end=(ax_val, 0.0)))
    panel.add(Dropline(start=(ax_val, ay_val), end=(0.0, ay_val)))
    panel.add(Vector((ax_val, 0.0), color=style.NEUTRAL_COLOR, alpha=0.95, zorder=2))
    panel.add(Vector((0.0, ay_val), color=style.NEUTRAL_COLOR, alpha=0.95, zorder=2))
    panel.add(Annotation("aₓ", xy=(ax_val / 2.0, -0.35),
                         color=style.PRIMARY_COLOR, fontweight="bold",
                         fontsize=style.DEFAULT_LABEL_FONT_SIZE,
                         ha="center", va="top"))
    panel.add(Annotation("aᵧ", xy=(ax_val + 0.30, ay_val / 2.0),
                         color=style.PRIMARY_COLOR, fontweight="bold",
                         fontsize=style.DEFAULT_LABEL_FONT_SIZE,
                         ha="left", va="center"))
    panel.add(Vector(A,
                     color=style.PRIMARY_COLOR,
                     label="a",
                     alpha=1.0,
                     zorder=3,
                     linewidth=style.DEFAULT_VECTOR_BOLD_LINEWIDTH))
    return panel


# --- col 1: DynamicPanel — reconstruction order -------------------------------
# 6 frames: build a tip-to-tail in two orders, both ending at A.
#   x-first ordering:
#     frame 0:  empty (just axes + ordering label)
#     frame 1:  x-leg from origin to (ax, 0)
#     frame 2:  y-leg appended, yields a = (ax, ay) (bold a on top)
#   y-first ordering:
#     frame 3:  y-leg from origin to (0, ay)
#     frame 4:  x-leg appended, yields a = (ax, ay) (bold a on top)
#     frame 5:  hold completed a (visual pause before loop restarts)
def _reconstruction_frames() -> list[list]:
    ax_val, ay_val = A
    primary = style.PRIMARY_COLOR
    neutral = style.NEUTRAL_COLOR

    def _x_leg(zorder=2):
        return Vector((ax_val, 0.0), color=neutral, alpha=0.95, zorder=zorder)

    def _y_leg_from_x_tip(zorder=2):
        return Vector((0.0, ay_val), origin=(ax_val, 0.0),
                      color=neutral, alpha=0.95, zorder=zorder)

    def _y_leg(zorder=2):
        return Vector((0.0, ay_val), color=neutral, alpha=0.95, zorder=zorder)

    def _x_leg_from_y_tip(zorder=2):
        return Vector((ax_val, 0.0), origin=(0.0, ay_val),
                      color=neutral, alpha=0.95, zorder=zorder)

    def _a_bold():
        return Vector(A,
                      color=primary, label="a",
                      alpha=1.0, zorder=4,
                      linewidth=style.DEFAULT_VECTOR_BOLD_LINEWIDTH)

    def _ordering_label(text):
        return Annotation(text, xy=(0.5, -0.10), transform="axes",
                          ha="center", va="top",
                          color=style.TICK_LABEL_COLOR,
                          fontsize=style.DEFAULT_LABEL_FONT_SIZE)

    return [
        [_ordering_label(r"$\hat{x}$ then $\hat{y}$")],
        [_x_leg(), _ordering_label(r"$\hat{x}$ then $\hat{y}$")],
        [_x_leg(), _y_leg_from_x_tip(), _a_bold(),
         _ordering_label(r"$\hat{x}$ then $\hat{y}$")],
        [_y_leg(), _ordering_label(r"$\hat{y}$ then $\hat{x}$")],
        [_y_leg(), _x_leg_from_y_tip(), _a_bold(),
         _ordering_label(r"$\hat{y}$ then $\hat{x}$")],
        [_y_leg(), _x_leg_from_y_tip(), _a_bold(),
         _ordering_label("same destination, either order")],
    ]


def _reconstruction_dynamic_panel() -> DynamicPanel:
    return DynamicPanel(
        frames=_reconstruction_frames(),
        interval_ms=750,
        repeat=True,
        lim=FOUND_LIM,
        axis_style="arrow",
        axis_labels=True,
        show_border=False,
        title="Tip-to-tail in either order",
        subtitle="x-first, then y-first — same arrival",
    )


# --- col 2: DynamicPanel — orthogonality --------------------------------------
# 5 frames: a is the FIXED REFERENCE anchored at (2, 3) on every frame.
# a' sweeps x through [+2, +1, 0, -1, -2] while a'.y stays at 3.
# The story: y is invariant under changes in x.
A_PRIME_X_SWEEP = [2.0, 1.0, 0.0, -1.0, -2.0]


def _orthogonality_frames() -> list[list]:
    ax_val, ay_val = A
    primary = style.PRIMARY_COLOR
    neutral = style.NEUTRAL_COLOR

    def _frame(apx_val: float) -> list:
        a_components = VectorComponents(
            A, first_axis="x", show_droplines=False,
            component_color=neutral,
        )
        ap_vec = (apx_val, ay_val)
        ap_components = VectorComponents(
            ap_vec, first_axis="x", show_droplines=False,
            component_color=neutral,
        )
        bold_a = Vector(A,
                        color=primary, label="a",
                        alpha=1.0, zorder=5,
                        linewidth=style.DEFAULT_VECTOR_BOLD_LINEWIDTH)
        bold_ap = Vector(ap_vec,
                         color=primary, label="a′",
                         alpha=1.0, zorder=5,
                         linewidth=style.DEFAULT_VECTOR_BOLD_LINEWIDTH)
        sweep_label = Annotation(
            f"a′ₓ = {apx_val:+.1f},  a′ᵧ = {ay_val:+.1f}",
            xy=(0.5, -0.10), transform="axes",
            ha="center", va="top",
            color=style.TICK_LABEL_COLOR,
            fontsize=style.DEFAULT_LABEL_FONT_SIZE,
        )
        return [a_components, ap_components, bold_a, bold_ap, sweep_label]

    return [_frame(apx) for apx in A_PRIME_X_SWEEP]


def _orthogonality_dynamic_panel() -> DynamicPanel:
    return DynamicPanel(
        frames=_orthogonality_frames(),
        interval_ms=900,
        repeat=True,
        lim=FOUND_LIM,
        axis_style="arrow",
        axis_labels=True,
        show_border=False,
        title="Orthogonality: y invariant under x",
        subtitle="a stays anchored at (2, 3); a′ sweeps x with y = 3",
    )


# --- compose the 1x3 mixed Figure ---------------------------------------------
panel_w = style.DEFAULT_PANEL_SIZE_INCHES * 1.3
panel_h = style.DEFAULT_PANEL_SIZE_INCHES * 1.4
fig = Figure(
    n_rows=1, n_cols=3,
    figsize=(panel_w * 3, panel_h),
    suptitle="§2.4 Figure 1 — Same components, either order, no cross-talk",
)
fig.add_panel(_static_xy_projection_panel(),    row=0, col=0)
fig.add_panel(_reconstruction_dynamic_panel(),  row=0, col=1)
fig.add_panel(_orthogonality_dynamic_panel(),   row=0, col=2)
fig.render()
plt.show()

# Optional GIF export of the orthogonality sweep (uncomment to enable):
# orth_panel = fig.panels[-1][0]   # (panel, row, col, rowspan, colspan, projection)
# orth_panel.save_gif("../../../assets/images/dsp/dsp_figure1_orthogonality_animation.gif", fps=2)
